# Week 9: Mapping from AI to Grasping

Name: Nguyen Phuc Minh Chau

Student ID: 22302421

**Lab Overview:** Eight progressive lab exercises guide students from raw pixel inputs through to a complete simulated grasp pipeline, culminating in hardware demo on Swift or Yahboom arm.

In [18]:
# Libraries
import numpy as np
from dataclasses import dataclass
from spatialmath import SE3
import roboticstoolbox as rtb

### Lab 9.1: Pixel to Viewing Ray

In [3]:
# TODO: Build the foundation function that converts a 2D pixel into a 3D direction vector in camera frame C.

def pixel_to_ray(K, u, v):
    uv1 = np.array([u, v, 1])  # Homogeneous coordinates
    # Apply inverse intrinsic matrix to get direction vector in camera frame
    x_C = np.linalg.inv(K) @ uv1
    # Normalize the direction vector
    dir_c = x_C / np.linalg.norm(x_C)
    return dir_c

### Lab 9.2: 3D Point in Camera Frame

In [8]:
# TODO: Given the 6D pose output from the AI network, construct a proper pose struct and verify consistency with the 2D detection.

@dataclass
class PoseC:
    R_C_0: np.ndarray   # shape (3, 3)
    t_C_0: np.ndarray   # shape (3,)

    def __post_init__(self):
        self.R_C_0 = np.asarray(self.R_C_0, dtype=float)
        self.t_C_0 = np.asarray(self.t_C_0, dtype=float).flatten()
        assert self.R_C_0.shape == (3, 3), "R_C_0 phải là ma trận 3×3"
        assert self.t_C_0.shape == (3,),   "t_C_0 phải là vector 3D"

def build_pose(R_C_0: np.ndarray, t_C_0: np.ndarray) -> PoseC:
    """Đóng gói output 6D của AI network vào PoseC struct."""
    return PoseC(R_C_0=R_C_0, t_C_0=t_C_0)

def project_to_pixel(K: np.ndarray, P_C: np.ndarray) -> tuple[float, float]:
    P_C = np.asarray(P_C, dtype=float).flatten()
    p   = K @ P_C           # (3,)
    return float(p[0] / p[2]), float(p[1] / p[2])

def verify_pose(
    K:     np.ndarray,
    pose:  PoseC,
    u_det: float,
    v_det: float,
    tol:   float = 2.0,
) -> dict:
    P_C = pose.t_C_0                        # = R @ 0 + t = t
    u_proj, v_proj = project_to_pixel(K, P_C)
    error = float(np.hypot(u_proj - u_det, v_proj - v_det))
    
    result = {
        "projected" : (round(u_proj, 2), round(v_proj, 2)),
        "detector"  : (u_det, v_det),
        "error_px"  : round(error, 3),
        "passed"    : error < tol,
    }

    status = "PASS" if result["passed"] else "FAIL"
    print(f"{status}  |  projected=({u_proj:.1f}, {v_proj:.1f})  "
          f"det=({u_det:.1f}, {v_det:.1f})  "
          f"err={error:.2f}px  (tol={tol}px)")
    return result


# ─────────────────────────────────────────────
# QUICK TEST
# ─────────────────────────────────────────────

if __name__ == "__main__":
    K = np.array([
        [800,   0, 320],
        [  0, 800, 240],
        [  0,   0,   1],
    ], dtype=float)

    R_C_0 = np.eye(3)
    t_C_0 = np.array([0.0, 0.0, 1.0])

    pose = build_pose(R_C_0, t_C_0)
    print("PoseC:", pose)

    result = verify_pose(K, pose, u_det=320.0, v_det=240.0)

PoseC: PoseC(R_C_0=array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]), t_C_0=array([0., 0., 1.]))
PASS  |  projected=(320.0, 240.0)  det=(320.0, 240.0)  err=0.00px  (tol=2.0px)


### Lab 9.3: Mapping Point p_C → p_B

In [9]:
#TODO: Implement the core mapping function using a pre-loaded calibration T_BC from a config file.

def cam_to_base(T_BC, p_C):
    pC_h =  np.r_[p_C, 1]  # homogeneous coordinates
    pB_h = T_BC.A @ pC_h
    return pB_h[:3]        #Extract the 3D position in base frame

### Lab 9.4: Mapping Full Pose T_CO → T_BO

In [10]:
def object_pose_in_base (T_BC, T_CO):
    return T_BC * T_CO # SE3 matrix multiplication

### Lab 9.5: Build T_target and Run IK

In [33]:
def build_target(T_BO, T_OT):
    T_BT = T_BO * T_OT
    return T_BT

def ik_solver(robot, T_target):
    sol = robot.ikine_LM(T_target)
    return sol.q

# Use IK from Week 8
T_BO = SE3(0.5, 0.0, 0.0)  # Example: object is 0.5m in front of the robot base
T_OT = SE3(0.0, 0.0, 0.1)  # Example: target grasp pose is 10cm above the object center
robot = rtb.models.URDF.Panda()

T_target = build_target(T_BO, T_OT)
q_target = ik_solver(robot, T_target)
print("Joint angles:", q_target)

Joint angles: [ 0.51556939  1.60731817 -0.35772054 -1.47376027 -1.72488759  0.36195378
  1.66239566]


### Lab 9.6: Full Pipeline with Simulated AI Data

In [34]:
# Simulated AI output: 3 objects on the table
objects = [SE3(0.15, 0.05, 0.0), # Object 1
           SE3(0.20, -0.03, 0.0), # Object 2
           SE3(0.10, 0.08, 0.0)] # Object 3

T_BC = SE3(0.1, 0.0, 0.5)  # Example: camera is 10cm to the right and 50cm above the robot base

for T_CO in objects:
    T_BO = T_BC * T_CO
    T_BT = T_BO * T_OT
    q = ik_solver(robot, T_BT)
    print(f"Grasp joint angles: {q}")

Grasp joint angles: [ 1.21962919 -1.74530932 -1.34340894 -3.0168921   1.77499453  1.77204528
 -0.31024162]
Grasp joint angles: [-0.80621942 -0.69137851  0.88664188 -2.96476655 -2.39329376  0.81279718
  1.40278513]
Grasp joint angles: [-1.34454603  0.66507815  2.04713075 -2.84292261 -0.82959108 -0.83824244
  2.48935372]


### Lab 9.7 — Effect of Calibration Errors

In [31]:
def corrupt_translation(T_BC, offset_m=(0.01, 0.01, 0.0)):
    return T_BC * SE3(*offset_m)

def corrupt_rotation(T_BC, deg=2.0, axis="z"):
    axes = {"x": SE3.Rx, "y": SE3.Ry, "z": SE3.Rz}
    return T_BC * axes[axis](np.deg2rad(deg))

def compare_grasp(T_BC_true, T_BC_corrupt, T_CO, T_OT=None):
    if T_OT is None:
        T_OT = SE3()                            # identity

    T_BO_true    = object_pose_in_base(T_BC_true,    T_CO)
    T_BO_corrupt = object_pose_in_base(T_BC_corrupt, T_CO)

    p_true    = build_target(T_BO_true,    T_OT).t
    p_corrupt = build_target(T_BO_corrupt, T_OT).t

    error = np.linalg.norm(p_corrupt - p_true) * 1000   # mm
    print(f"  Grasp (true)    : {p_true.round(4)}")
    print(f"  Grasp (corrupt) : {p_corrupt.round(4)}")
    print(f"  Error           : {error:.2f} mm")
    print(f"  Direction shift : {(p_corrupt - p_true).round(4)}")
    return error

#### Demo

In [32]:
# T_BC: camera gắn trên base, lệch 0.5m theo Z, xoay 90° quanh Z
T_BC = SE3(0.0, 0.0, 0.5) * SE3.Rz(np.deg2rad(90))

# Object cách camera 0.3m theo trục Z của camera
T_CO = SE3(0.0, 0.0, 0.3)

print("=" * 50)
print("▶ Translation Error (+1cm x, +1cm y)")
print("=" * 50)
T_BC_t = corrupt_translation(T_BC, offset_m=(0.01, 0.01, 0.0))
compare_grasp(T_BC, T_BC_t, T_CO)

print()
print("=" * 50)
print("▶ Rotation Error (+2° around Z)")
print("=" * 50)
T_BC_r = corrupt_rotation(T_BC, deg=2.0, axis="z")
compare_grasp(T_BC, T_BC_r, T_CO)

print()
print("=" * 50)
print("▶ Rotation Error — object xa hơn (0.8m)")
print("=" * 50)
T_CO_far = SE3(0.0, 0.0, 0.8)
compare_grasp(T_BC, T_BC_r, T_CO_far)

▶ Translation Error (+1cm x, +1cm y)
  Grasp (true)    : [0.  0.  0.8]
  Grasp (corrupt) : [-0.01  0.01  0.8 ]
  Error           : 14.14 mm
  Direction shift : [-0.01  0.01  0.  ]

▶ Rotation Error (+2° around Z)
  Grasp (true)    : [0.  0.  0.8]
  Grasp (corrupt) : [0.  0.  0.8]
  Error           : 0.00 mm
  Direction shift : [0. 0. 0.]

▶ Rotation Error — object xa hơn (0.8m)
  Grasp (true)    : [0.  0.  1.3]
  Grasp (corrupt) : [0.  0.  1.3]
  Error           : 0.00 mm
  Direction shift : [0. 0. 0.]


np.float64(0.0)